# 01 — Customer Consumption · Short Term

Materialises the bronze → silver → gold tables in
[`../specifications/01-customer-consumption-short-term.md`](../specifications/01-customer-consumption-short-term.md).

**Depends on 04** — reads the curated `volume_forecast_silver_meter_profile` and the shared dimensions.

**Capability tables**
- `volume_forecast_bronze_weather_obs` (temperature / irradiance / cloud drivers)
- `volume_forecast_silver_consumption_st` (probabilistic P10/P50/P90 demand per segment × interval)
- `volume_forecast_gold_consumption_st_summary` (headline KPIs per zone per day)

**Probabilistic note:** the forecast is anchored on the curated net-load profile (04) with a small bias and a
lead-time-widening band so the app can show P10/P50/P90. `forecast_ts` carries the issue time so the page
can show "as last updated".

**UC comments:** [`uc_table_comments.py`](./uc_table_comments.py) — applied in the final cell.

In [ ]:
import os
import math
import random
import datetime as dt

from pyspark.sql import functions as F
from pyspark.sql import Row
from pyspark.sql.window import Window

dbutils.widgets.text("catalog", os.environ.get("DEMO_UC_CATALOG", "energy_utilities"))
dbutils.widgets.text("schema", os.environ.get("DEMO_UC_SCHEMA", "energy_trading2"))

CATALOG = dbutils.widgets.get("catalog").strip() or "energy_utilities"
SCHEMA = dbutils.widgets.get("schema").strip() or "energy_trading2"
print(f"Target: {CATALOG}.{SCHEMA}")
spark.sql(f"USE `{CATALOG}`.`{SCHEMA}`")


def fq(name: str) -> str:
    return f"`{CATALOG}`.`{SCHEMA}`.`{name}`"


random.seed(1)

TODAY = dt.date.today()
# Rolling horizon: 2 history days, today (half settled), 2 forecast days (matches 04).
DELIVERY_DATES = [TODAY + dt.timedelta(days=d) for d in (-2, -1, 0, 1, 2)]
N_INTERVALS = 96
NOW_INDEX = 56  # ~14:00 "now" cursor on today; earlier (and prior days) = settled / realised
NOW_TS = dt.datetime.combine(TODAY, dt.time(0, 0)) + dt.timedelta(minutes=15 * NOW_INDEX)
SNAP_TS = dt.datetime.combine(TODAY, dt.time(15, 0))
ZONES = ["DE", "NL", "FR", "BE", "AT"]

# Per-zone temperature offset (deg C) so the weather driver varies across zones.
ZONE_TEMP_OFFSET = {"DE": 0.0, "NL": 0.5, "FR": 2.0, "BE": 0.5, "AT": -1.5}

# Two forecast runs (rolling re-forecast). capture = fraction of the weather swing the
# model gets right; the later run is sharper and closer to delivery.
RUN_PREV_TS = NOW_TS - dt.timedelta(hours=6)
RUN_LATEST_TS = NOW_TS

# Weather sensitivity by segment (residential most weather-driven; heavy industry least).
SEG_WEATHER_SENS = {"RES": 0.9, "RES_PV": 0.9, "RES_EV": 1.0, "SME": 0.5, "CNI": 0.15}


def interval_ts(day: dt.date, idx: int) -> dt.datetime:
    return dt.datetime.combine(day, dt.time(0, 0)) + dt.timedelta(minutes=15 * idx)


def is_settled(day: dt.date, idx: int) -> bool:
    if day < TODAY:
        return True
    if day == TODAY:
        return idx < NOW_INDEX
    return False


def season_of(day: dt.date) -> str:
    m = day.month
    if m in (12, 1, 2):
        return "WINTER"
    if m in (3, 4, 5):
        return "SPRING"
    if m in (6, 7, 8):
        return "SUMMER"
    return "AUTUMN"


# Seasonal-normal temperature (deg C) — the heating/cooling reference for the anomaly.
_T_NORM = {"WINTER": 3.0, "SPRING": 12.0, "SUMMER": 22.0, "AUTUMN": 12.0}


def normal_temp(idx: int, season: str, zone: str) -> float:
    return _T_NORM[season] + 5.0 * math.sin((idx / 96.0 - 0.30) * 2 * math.pi) + ZONE_TEMP_OFFSET[zone]


def weather_factor(temp: float, nrm: float) -> float:
    # Demand rises when it is colder (heating) or hotter (cooling) than the seasonal normal.
    dev = temp - nrm
    return 1.0 + 0.012 * max(0.0, -dev) + 0.010 * max(0.0, dev)


# Curated net-load profile from notebook 04 is the climatological anchor for the forecast.
assert spark.catalog.tableExists(fq("volume_forecast_silver_meter_profile")), \
    "Run 04_smart_metering first — volume_forecast_silver_meter_profile is missing."

In [ ]:
# ---- Bronze: weather drivers (per-zone temperature anomaly regime) ----
# One temperature regime per (day, zone) so a given day runs colder/warmer than normal;
# this anomaly is what makes demand deviate from the climatological profile.
_temp_regime = {(day, z): random.gauss(0.0, 3.0) for day in DELIVERY_DATES for z in ZONES}

weather_rows = []
wx: dict[tuple, tuple] = {}  # (day, zone, idx) -> (temperature_c, normal_temp)
for day in DELIVERY_DATES:
    season = season_of(day)
    fts = dt.datetime.combine(day, dt.time(6, 0))  # weather feed issued 06:00
    for z in ZONES:
        for idx in range(N_INTERVALS):
            nrm = normal_temp(idx, season, z)
            temp = nrm + _temp_regime[(day, z)] + random.gauss(0, 0.8)
            h = idx * 15 / 60.0
            ghi = max(0.0, math.sin(math.pi * (h - 6.0) / 13.0)) * 900.0 if 6 < h < 19 else 0.0
            cloud = max(0.0, min(100.0, (1 - ghi / 900.0) * 55 + random.gauss(0, 12))) if ghi > 0 else random.uniform(20, 80)
            wx[(day, z, idx)] = (temp, nrm)
            weather_rows.append(Row(
                ingestion_ts=fts, forecast_ts=fts,
                source=random.choice(["ECMWF", "GFS", "ICON", "VENDOR_X"]),
                zone_code=z, interval_start=interval_ts(day, idx),
                temperature_c=round(temp, 1),
                solar_irradiance_wm2=round(ghi, 1), cloud_cover_pct=round(cloud, 1),
            ))
spark.createDataFrame(weather_rows).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(fq("volume_forecast_bronze_weather_obs"))

# ---- Silver: probabilistic consumption — climatological profile shaped by weather ----
# actual_mw  = profile * full weather response (settled intervals only; null in the future)
# p50_mw     = profile * (captured weather response) + run noise; the later run captures more
# Two vintages per interval (prev run + latest run) give a real forecast-move signal and
# let 08 backtest the latest vintage against actuals.
from pyspark.sql.types import (StructType, StructField, StringType, DoubleType,
                               TimestampType, DateType)

cons_schema = StructType([
    StructField("delivery_date", DateType()),
    StructField("interval_start", TimestampType()),
    StructField("forecast_ts", TimestampType()),
    StructField("zone_code", StringType()),
    StructField("segment_code", StringType()),
    StructField("p10_mw", DoubleType()),
    StructField("p50_mw", DoubleType()),
    StructField("p90_mw", DoubleType()),
    StructField("prev_p50_mw", DoubleType()),
    StructField("forecast_delta_mw", DoubleType()),
    StructField("temperature_c", DoubleType()),
    StructField("actual_mw", DoubleType()),
    StructField("dq_status", StringType()),
])

profile = spark.table(fq("volume_forecast_silver_meter_profile")).collect()
cons_rows = []
for p in profile:
    midnight = dt.datetime.combine(p.delivery_date, dt.time(0, 0))
    idx = int(round((p.interval_start - midnight).total_seconds() / 900.0))
    base = float(p.net_load_mw)
    temp, nrm = wx[(p.delivery_date, p.zone_code, idx)]
    sens = SEG_WEATHER_SENS.get(p.segment_code, 0.5)
    swing = sens * (weather_factor(temp, nrm) - 1.0)  # weather-driven fraction of demand

    settled = is_settled(p.delivery_date, idx)
    actual = base * (1.0 + swing) if settled else None

    p50_prev = base * (1.0 + 0.70 * swing) * (1.0 + random.gauss(0, 0.020))
    p50_latest = base * (1.0 + 0.90 * swing) * (1.0 + random.gauss(0, 0.008))

    def _band(p50: float, vts: dt.datetime) -> tuple:
        lead_h = max(0.0, (p.interval_start - vts).total_seconds() / 3600.0)
        sigma = base * (0.015 + 0.010 * math.sqrt(lead_h)) if lead_h > 0 else base * 0.004
        return max(0.0, p50 - 1.2816 * sigma), p50 + 1.2816 * sigma

    for vts, p50v, prevv in ((RUN_PREV_TS, p50_prev, None), (RUN_LATEST_TS, p50_latest, p50_prev)):
        lo, hi = _band(p50v, vts)
        cons_rows.append((
            p.delivery_date, p.interval_start, vts, p.zone_code, p.segment_code,
            round(lo, 3), round(p50v, 3), round(hi, 3),
            (round(prevv, 3) if prevv is not None else None),
            (round(p50v - prevv, 3) if prevv is not None else None),
            round(temp, 1),
            (round(actual, 3) if actual is not None else None),
            p.dq_status,
        ))
spark.createDataFrame(cons_rows, cons_schema).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(fq("volume_forecast_silver_consumption_st"))

print("weather:", spark.table(fq("volume_forecast_bronze_weather_obs")).count())
print("consumption_st:", spark.table(fq("volume_forecast_silver_consumption_st")).count())

In [ ]:
# ---- Gold: zonal headline KPIs per day (latest forecast vintage only) ----
cons_all = spark.table(fq("volume_forecast_silver_consumption_st"))
latest_ts = cons_all.agg(F.max("forecast_ts").alias("m")).first()["m"]
cons = cons_all.filter(F.col("forecast_ts") == F.lit(latest_ts))

# Zonal demand per interval (sum across segments) and the uncertainty band.
zint = cons.groupBy("delivery_date", "zone_code", "interval_start").agg(
    F.sum("p50_mw").alias("z_p50"),
    F.sum(F.col("p90_mw") - F.col("p10_mw")).alias("z_band"),
)

wspec = Window.partitionBy("delivery_date", "zone_code").orderBy(F.col("z_p50").desc())
peak = (zint.withColumn("rn", F.row_number().over(wspec)).filter("rn = 1")
        .select("delivery_date", "zone_code",
                F.col("interval_start").alias("peak_interval"),
                F.round(F.col("z_p50"), 2).alias("peak_mw")))

agg = zint.groupBy("delivery_date", "zone_code").agg(
    F.round(F.avg("z_p50"), 2).alias("avg_demand_mw"),         # mean instantaneous demand (MW)
    F.round(F.avg("z_band"), 2).alias("band_width_mw"),
)

# Daily energy total (MWh) = sum of P50 power over all segments/intervals x 0.25 h.
energy = cons.groupBy("delivery_date", "zone_code").agg(
    F.round(F.sum("p50_mw") * 0.25, 1).alias("total_demand_mwh")
)

dq = cons.groupBy("delivery_date", "zone_code").agg(
    F.max(F.when(F.col("dq_status") == "SUSPECT", 2).when(F.col("dq_status") == "IMPUTED", 1).otherwise(0)).alias("dqr")
)

summary = (agg
    .join(energy, ["delivery_date", "zone_code"])
    .join(peak, ["delivery_date", "zone_code"])
    .join(dq, ["delivery_date", "zone_code"])
    .withColumn("snapshot_ts", F.lit(SNAP_TS).cast("timestamp"))
    .withColumn("dq_status", F.when(F.col("dqr") == 2, "SUSPECT").when(F.col("dqr") == 1, "IMPUTED").otherwise("GOOD"))
    .withColumn("headline", F.when(F.col("band_width_mw") > 40, F.lit("Wide uncertainty band — weather-driven demand risk"))
                             .otherwise(F.lit("Demand forecast within normal band")))
    .select("delivery_date", "zone_code", "snapshot_ts", "headline",
            "avg_demand_mw", "total_demand_mwh", "peak_mw", "peak_interval", "band_width_mw", "dq_status"))

summary.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(fq("volume_forecast_gold_consumption_st_summary"))
display(spark.table(fq("volume_forecast_gold_consumption_st_summary")).orderBy(F.col("delivery_date").desc(), "zone_code"))

In [ ]:
# Row counts + Unity Catalog comments.
for t in [
    "volume_forecast_bronze_weather_obs",
    "volume_forecast_silver_consumption_st",
    "volume_forecast_gold_consumption_st_summary",
]:
    print(f"  {t:44s}  {spark.table(fq(t)).count():>10,} rows")

from pathlib import Path

_uc_paths = []
try:
    _nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    _uc_paths.append(Path(_nb).parent / "uc_table_comments.py")
except Exception:
    pass
_uc_paths.append(Path.cwd() / "uc_table_comments.py")

_uc_py = next((p for p in _uc_paths if p.is_file()), None)
if _uc_py is None:
    raise FileNotFoundError("uc_table_comments.py not found next to this notebook.")

exec(_uc_py.read_text(), globals())
apply_volume_forecast_notebook_01_comments(spark, CATALOG, SCHEMA)
print("UC comments applied for notebook 01.")